In [4]:
import pandas as pd
import geopandas as gpd
import folium
import pyarrow.parquet as pq
import os
import json
from io import StringIO
import branca.colormap as cm
import numpy as np

# ==============================================================================
# STATISCHE KAART MET NIEUW SHAPEFILE (wegen_in_out.shp)
# ==============================================================================

# --- Configuratie ---
TOMTOM_FILE_PATH = os.path.join('Data', '20250820163000_stream.tomtom.analyze-sail.parquet')
# AANGEPAST: Pad naar het nieuwe shapefile in dezelfde map als het script
SHAPEFILE_PATH = os.path.join('Data', 'NWB_roads', 'NWB_roads', 'wegen_in_out.shp')
SAMPLE_FRACTION = 0.5 # Behoud 50% sample
# --- !! BELANGRIJK !! ---
# Pas deze variabele aan nadat u de kolomnamen hieronder heeft gezien!
# Zoek naar de kolom die overeenkomt met de 'id' uit de TomTom data.
SHAPEFILE_ID_COL = 'wvk_id'



def parse_tomtom_data(value_string):
    """Verwerkt de data uit de '_value' kolom."""
    try:
        start_index = value_string.find('{')
        if start_index == -1: return None
        data = json.loads(value_string[start_index:])
        if 'data' in data and isinstance(data['data'], str):
            return pd.read_csv(StringIO(data['data']))
    except (json.JSONDecodeError, KeyError):
        return None
    return None

def create_traffic_map():
    """Voert het volledige proces uit van data laden tot het maken van een Folium kaart."""
    try:
        # --- 1. Data Laden, Verwerken en Aggregeren ---
        print(f"Laden van {SAMPLE_FRACTION*100}% sample uit {TOMTOM_FILE_PATH}...")
        df_raw = pd.read_parquet(TOMTOM_FILE_PATH).sample(frac=SAMPLE_FRACTION, random_state=1)
        
        print("Verwerken van verkeersdata...")
        parsed_dfs = df_raw['_value'].apply(parse_tomtom_data)
        df_traffic = pd.concat(parsed_dfs.dropna().tolist(), ignore_index=True)
        
        print("Aggregeren van verkeersdata...")
        df_traffic_agg = df_traffic.groupby('id')['traffic_level'].mean().reset_index()
        print(f"Data geaggregeerd naar {len(df_traffic_agg):,} unieke wegvakken.")

        # --- 2. Kaart Laden en Voorbereiden ---
        print(f"\nLaden van kaartdata: {SHAPEFILE_PATH}...")
        gdf = gpd.read_file(SHAPEFILE_PATH)
        print("Kaartdata succesvol geladen!")
        
        # Controleer of de ID kolom bestaat
        if SHAPEFILE_ID_COL not in gdf.columns:
             print(f"\n--- FOUT ---")
             print(f"De kolom '{SHAPEFILE_ID_COL}' is NIET GEVONDEN in {SHAPEFILE_PATH}.")
             print(f"Beschikbare kolommen zijn: {gdf.columns.tolist()}")
             return None
             
        print("Coördinatensysteem omzetten...")
        gdf = gdf.to_crs(epsg=4326)
        
        print("Filteren op Amsterdam...")
        min_lon, min_lat, max_lon, max_lat = 4.72, 52.28, 5.08, 52.43
        # Gebruik .clip() voor een robuustere filter
        gdf_amsterdam = gdf.clip([min_lon, min_lat, max_lon, max_lat]).copy()
        print(f"Filteren voltooid: {len(gdf_amsterdam):,} wegvakken in regio.")

        # --- 3. Data Koppelen met 'inner' Merge ---
        print("\nKoppelen van data met een 'inner' merge...")
        # Robuuste typeconversie
        gdf_amsterdam[SHAPEFILE_ID_COL] = pd.to_numeric(gdf_amsterdam[SHAPEFILE_ID_COL], errors='coerce').astype('Int64').astype(str)
        df_traffic_agg['id'] = pd.to_numeric(df_traffic_agg['id'], errors='coerce').astype('Int64').astype(str)
        
        # Gebruik 'inner' om alleen gematchte wegen te behouden.
        merged_gdf = gdf_amsterdam.merge(df_traffic_agg, left_on=SHAPEFILE_ID_COL, right_on='id', how='inner')
        print(f"Succesvol {len(merged_gdf):,} wegvakken gekoppeld en voorbereid voor de kaart.")

        if merged_gdf.empty:
            print("\nWAARSCHUWING: Geen data gekoppeld. Controleer de ID kolommen.")
            return None

        # --- Opschonen van Datumkolommen (voor de zekerheid) ---
        print("Opschonen van datumkolommen...")
        for col in merged_gdf.columns:
            if pd.api.types.is_datetime64_any_dtype(merged_gdf[col]):
                merged_gdf[col] = merged_gdf[col].astype(str)

        # --- 4. Visualisatie met Folium GeoJson en Style Function ---
        print("Genereren van de Folium verkeerskaart met directe styling...")
        
        amsterdam_location = [52.3676, 4.9041]
        m = folium.Map(location=amsterdam_location, zoom_start=12, tiles="CartoDB positron")

        # Maak een kleurenschaal
        colormap = cm.LinearColormap(colors=['green', 'yellow', 'red'], vmin=0, vmax=1)
        colormap.caption = 'Gemiddelde Verkeersdrukte'

        # Definieer de style functie die voor ELK wegvak de kleur bepaalt
        def style_function(feature):
            traffic = feature['properties']['traffic_level']
            return {
                'fillOpacity': 0.8,
                'weight': 1.5,
                 # Gebruik de colormap direct. NaN wordt automatisch overgeslagen door 'inner' merge.
                'fillColor': colormap(traffic) if not pd.isna(traffic) else '#d3d3d3', # Grijs als fallback
                'color': colormap(traffic) if not pd.isna(traffic) else '#d3d3d3'      # Grijs als fallback
            }

        # Maak een GeoJson laag met de tooltip en de style functie
        folium.GeoJson(
            merged_gdf, # Nu alleen de 9597 gematchte rijen
            style_function=style_function,
            tooltip=folium.GeoJsonTooltip(
                # Toon relevante velden, inclusief traffic_level
                fields=[SHAPEFILE_ID_COL, 'traffic_le', 'traffic_level'], 
                aliases=['Wegvak ID:', 'Origineel Traffic Level:', 'Gem. Traffic Level:'],
                sticky=True
            )
        ).add_to(m)

        # Voeg de legenda toe aan de kaart
        m.add_child(colormap)
        
        print("\n--- SUCCES! ---")
        print("De kaart is aangemaakt en zou nu moeten verschijnen.")
        
        m.save("verkeerskaart_amsterdam_wegen_in_out_styled.html")
        print("Een kopie is opgeslagen als 'verkeerskaart_amsterdam_wegen_in_out_styled.html'")
        
        return m

    except Exception as e:
        print(f"Er is een onverwachte fout opgetreden: {e}")
        return None

# Voer het script uit en toon de kaart
traffic_map = create_traffic_map()
# Alleen weergeven als de kaart succesvol is aangemaakt
if traffic_map:
    from IPython.display import display 
    display(traffic_map)


Laden van 50.0% sample uit Data\20250820163000_stream.tomtom.analyze-sail.parquet...
Verwerken van verkeersdata...
Aggregeren van verkeersdata...
Data geaggregeerd naar 10,742 unieke wegvakken.

Laden van kaartdata: Data\NWB_roads\NWB_roads\wegen_in_out.shp...
Kaartdata succesvol geladen!
Coördinatensysteem omzetten...
Filteren op Amsterdam...
Filteren voltooid: 9,603 wegvakken in regio.

Koppelen van data met een 'inner' merge...
Succesvol 9,603 wegvakken gekoppeld en voorbereid voor de kaart.
Opschonen van datumkolommen...
Genereren van de Folium verkeerskaart met directe styling...

--- SUCCES! ---
De kaart is aangemaakt en zou nu moeten verschijnen.
Een kopie is opgeslagen als 'verkeerskaart_amsterdam_wegen_in_out_styled.html'
